##Importing libraries

In [0]:
import requests
import json
from pyspark.sql import SparkSession

##Call The API

In [0]:
url = "https://api.coingecko.com/api/v3/coins/markets"

params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 100,
    "page": 1,
    "sparkline": False
}

api_key = dbutils.secrets.get(scope="crypto-pipeline", key="coingecko-api-key")

headers = {
    "x-cg-demo-api-key": api_key
}

response = requests.get(url, headers=headers, params=params)
data = response.json()

##Save To Volume 

In [0]:
raw_path = "/Volumes/workspace/bronze/raw_sources/crypto_raw.json"

with open(raw_path, "w") as f:
    json.dump(data, f)

print("Raw JSON saved to Volume ")

##Read and Save as Delta Table

In [0]:
df = spark.read.option("multiline", "true").json(raw_path)

df.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.crypto_raw")

print("Delta Table created successfully!")